Agentic RAG

In [1]:
import os
import logging
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [2]:
os.environ['ANONYMIZED_TELEMETRY'] = 'False' 
logging.getLogger('httpx').setLevel(logging.WARNING)

In [3]:
load_dotenv()

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

Load the Vector Store

In [4]:
persist_dir = r'C:\Users\USER\rag_course\chroma_db_domain'

# Load the store
embeddings = OpenAIEmbeddings()
vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings,
)

# Base retriever — top 4 by similarity
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
print('Vector store loaded')
print(f'    Chunks in store: {vectorstore._collection.count()}')
print('Retriever k: 4')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store loaded
    Chunks in store: 80
Retriever k: 4


The Classifier Prompt

In [5]:
classifier_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You decide if a user question needs the knowledge base.\n'
     '\n'
     'CATEGORIES:\n'
     '- RETRIEVE: the question is about crops, plant diseases, treatments, '
     'or Nigeria public health facts. These need the knowledge base.\n'
     '- DIRECT: general knowledge, greetings, math, chit-chat, or anything '
     'the LLM can answer on its own. These do NOT need the knowledge base.\n'
     '\n'
     'EXAMPLES:\n'
     '- "How do I treat cassava mosaic disease?"  → RETRIEVE\n'
     '- "What is maize smut?"                    → RETRIEVE\n'
     '- "Hello, how are you?"                    → DIRECT\n'
     '- "What is 2 + 2?"                         → DIRECT\n'
     '- "What is the capital of France?"         → DIRECT\n'
     '- "What are the top causes of death in Nigeria?" → RETRIEVE\n'
     '\n'
     'Respond with exactly one word: RETRIEVE or DIRECT. No explanation.'),
    ('human', '{question}'),
])

print('Classifier prompt ready')


Classifier prompt ready


The Classifier Chain

In [6]:
classifier_chain = classifier_prompt | llm | StrOutputParser()

Test the Classifier on Its Own

In [7]:
test_questions = [
    'How do I treat cassava mosaic disease?',
    'What is maize smut?',
    'Hello, how are you?',
    'What is 2 + 2?',
    'What is the capital of France?',
    'What are the top causes of death in Nigeria?',
]

for q in test_questions:
    result = classifier_chain.invoke({'question': q}).strip().upper()
    print(f'{result:<10} <- {q}')

RETRIEVE   <- How do I treat cassava mosaic disease?
RETRIEVE   <- What is maize smut?
DIRECT     <- Hello, how are you?
DIRECT     <- What is 2 + 2?
DIRECT     <- What is the capital of France?
RETRIEVE   <- What are the top causes of death in Nigeria?


The Direct Answer Chain

In [8]:
direct_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a helpful assistant. '
     'Answer the user\'s question directly using your own knowledge. '
     'Be concise and clear.'),
    ('human', '{question}'),
])

direct_chain = direct_prompt | llm | StrOutputParser()

print('Direct answer chain ready')

Direct answer chain ready


Test It Directly

In [9]:
print(direct_chain.invoke({'question': 'What is 2 + 2?'}))
print()
print(direct_chain.invoke({'question': 'Hello, how are you?'}))

2 + 2 equals 4.

Hello! I'm just a program, but I'm here and ready to help you. How can I assist you today?


The Retrieval Answer Chain

In [10]:
retrieval_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a helpful assistant. '
     'Answer the user\'s question using ONLY the context provided below. '
     'If the context does not contain the answer, say "I don\'t know".'),
    ('human', 'Context:\n{context}\n\nQuestion: {question}'),
])

retrieval_chain = retrieval_prompt | llm | StrOutputParser()

print('Retrieval answer chain ready')

Retrieval answer chain ready


Test It With Real Context

In [11]:
# Get some chunks from the store
docs = retriever.invoke('cassava mosaic disease')
context = '\n\n'.join(d.page_content for d in docs)

# Ask the retrieval chain
answer = retrieval_chain.invoke({
    'context': context,
    'question': 'How do I treat cassava mosaic disease?',
})

print('Answer:')
print(answer)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Answer:
To treat cassava mosaic disease, use disease-free cuttings, plant resistant varieties, and remove infected plants early.


Self-RAG Router

In [12]:
def self_rag(question: str, verbose: bool = True) -> str:
    '''
    Route the question, then answer via the right chain
    '''
    
    # Step 1: Classify
    category = classifier_chain.invoke({'question': question}).strip().upper()
    
    if verbose:
        print(f'Classifier -> {category}')
        
    # Step 2: Route
    if 'RETRIEVE' in category:
        # Retrieval path
        docs = retriever.invoke(question)
        context = '\n\n'.join(d.page_content for d in docs)
        
        if verbose:
            print(f'Retrieved {len(docs)} chunks')
        return retrieval_chain.invoke({
            'context': context,
            'question': question,
        })
        
    else:
        # Direct path
        if verbose:
            print('Answering directly (no retrieval)')
        return direct_chain.invoke({'question': question})
    
print('Self-RAG router ready')

Self-RAG router ready


Test the Router

In [15]:
questions = [
    'Hello!',
    'What is 2 + 2?',
    'What is the capital of France?',
    'How do I treat cassava mosaic disease?',
    'What is maize smut?',
    'What are the top causes of death in Nigeria?',
    'Who is present Nigeria president and how old is he/she, what is his/her party? Is it a male or female?/ And also tell me both in English, in Yoruba an what is the chemical formula for potash?'
]

for q in questions:
    print(f'\n❓ {q}')
    answer = self_rag(q)
    print(f'{answer}')
    print('-' * 70)


❓ Hello!
Classifier -> DIRECT
Answering directly (no retrieval)
Hello! How can I assist you today?
----------------------------------------------------------------------

❓ What is 2 + 2?
Classifier -> DIRECT
Answering directly (no retrieval)
2 + 2 equals 4.
----------------------------------------------------------------------

❓ What is the capital of France?
Classifier -> DIRECT
Answering directly (no retrieval)
The capital of France is Paris.
----------------------------------------------------------------------

❓ How do I treat cassava mosaic disease?
Classifier -> RETRIEVE
Retrieved 4 chunks
To treat cassava mosaic disease, you should use disease-free cuttings, plant resistant varieties, and remove infected plants early.
----------------------------------------------------------------------

❓ What is maize smut?
Classifier -> RETRIEVE
Retrieved 4 chunks
Maize smut is a disease affecting maize, characterized by large grey or black galls on ears, stalks, and leaves. The control 

In [14]:
edge_cases = [
    # --- Ambiguous: mentions crops but isn't asking for data ---
    'Do you like cassava?',
    'Is farming hard?',

    # --- Domain-looking but general knowledge ---
    'How many countries are in Africa?',
    'What is photosynthesis?',

    # --- Out of scope but tricky ---
    'Who is the president of Nigeria?',
    'What is blockchain?',

    # --- Deep domain questions ---
    'What percentage of deaths are from malaria?',
    'What is tomato leaf curl virus?',

    # --- Conversational follow-ups (no context) ---
    'Thanks!',
    'Tell me more about that.',
]

for q in edge_cases:
    print(f'\n❓ {q}')
    answer = self_rag(q)
    print(f'💬 {answer[:150]}')
    print('-' * 70)



❓ Do you like cassava?
Classifier -> DIRECT
Answering directly (no retrieval)
💬 As an AI, I don't have personal preferences or tastes. However, cassava is a versatile root vegetable that is enjoyed by many people around the world 
----------------------------------------------------------------------

❓ Is farming hard?
Classifier -> DIRECT
Answering directly (no retrieval)
💬 Yes, farming can be hard work. It often involves long hours, physical labor, and dealing with various challenges such as weather conditions, pests, an
----------------------------------------------------------------------

❓ How many countries are in Africa?
Classifier -> DIRECT
Answering directly (no retrieval)
💬 As of now, there are 54 recognized countries in Africa.
----------------------------------------------------------------------

❓ What is photosynthesis?
Classifier -> DIRECT
Answering directly (no retrieval)
💬 Photosynthesis is the process by which green plants, algae, and some bacteria convert light e